In [ ]:
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

In [ ]:
def crear_mosaico_aladin_style():
    """
    Crea un mosaico usando las coordenadas RA/DEC reales para posicionar los campos
    """
    print("🗺️ Creando mosaico con coordenadas reales...")
    
    campos = ['CenA01', 'CenA02', 'CenA03', 'CenA04', 'CenA05', 'CenA06']
    
    # Obtener las coordenadas reales de cada campo
    coordenadas = {}
    for campo in campos:
        try:
            with fits.open(f"../anac_data/{campo}/{campo}_F861.fits.fz") as hdul:
                header = hdul[1].header
                ra = header['CRVAL1']
                dec = header['CRVAL2']
                coordenadas[campo] = (ra, dec)
                print(f"   {campo}: RA={ra:.4f}, DEC={dec:.4f}")
        except Exception as e:
            print(f"❌ Error obteniendo coordenadas de {campo}: {e}")
    
    if len(coordenadas) < 2:
        print("❌ No hay suficientes campos con coordenadas")
        return None
    
    # Analizar la distribución espacial de los campos
    ras = [coord[0] for coord in coordenadas.values()]
    decs = [coord[1] for coord in coordenadas.values()]
    
    ra_min, ra_max = min(ras), max(ras)
    dec_min, dec_max = min(decs), max(decs)
    
    print(f"📊 Rango RA: {ra_min:.4f} a {ra_max:.4f}")
    print(f"📊 Rango DEC: {dec_min:.4f} a {dec_max:.4f}")
    print(f"📊 Tamaño del área: {ra_max-ra_min:.2f}° × {dec_max-dec_min:.2f}°")
    
    # Crear un mosaico más grande para acomodar todos los campos
    mosaic_size = 4000
    mosaic_r = np.zeros((mosaic_size, mosaic_size))
    mosaic_g = np.zeros((mosaic_size, mosaic_size))
    mosaic_b = np.zeros((mosaic_size, mosaic_size))
    
    # Calcular posiciones basadas en coordenadas reales
    # En astronomía, RA aumenta hacia el este (izquierda en imágenes) y DEC hacia el norte (arriba)
    posiciones = {}
    for campo, (ra, dec) in coordenadas.items():
        # Normalizar coordenadas al rango [0, mosaic_size]
        # Invertir RA porque en imágenes el este suele estar a la izquierda
        x = mosaic_size - ((ra - ra_min) / (ra_max - ra_min)) * (mosaic_size - 1200)
        y = ((dec - dec_min) / (dec_max - dec_min)) * (mosaic_size - 1200)
        
        # Ajustar para centrar mejor
        x = int(x + 600)  # Margen izquierdo
        y = int(y + 600)  # Margen inferior
        
        posiciones[campo] = (x, y)
        print(f"   📍 {campo} → posición mosaico: ({x}, {y})")
    
    print("🎯 Colocando campos en el mosaico...")
    
    for campo in tqdm(campos, desc="Procesando campos"):
        try:
            if campo not in posiciones:
                continue
                
            x_center, y_center = posiciones[campo]
            
            # Cargar subregiones más grandes para mejor solapamiento
            sub_size = 1200  # Aumentado de 800 a 1200
            
            def cargar_subregion(campo, filtro):
                with fits.open(f"../anac_data/{campo}/{campo}_{filtro}.fits.fz") as hdul:
                    data_full = hdul[1].data
                    center_y, center_x = data_full.shape[0]//2, data_full.shape[1]//2
                    half_size = sub_size // 2
                    
                    # Asegurar que no nos salimos de los límites
                    start_y = max(0, center_y - half_size)
                    end_y = min(data_full.shape[0], center_y + half_size)
                    start_x = max(0, center_x - half_size)
                    end_x = min(data_full.shape[1], center_x + half_size)
                    
                    return data_full[start_y:end_y, start_x:end_x]
            
            data_r = cargar_subregion(campo, 'F861')
            data_g = cargar_subregion(campo, 'F660')
            data_b = cargar_subregion(campo, 'F515')
            
            # Calcular posición en el mosaico
            y_start = max(0, y_center - sub_size//2)
            y_end = min(mosaic_size, y_center + sub_size//2)
            x_start = max(0, x_center - sub_size//2)
            x_end = min(mosaic_size, x_center + sub_size//2)
            
            # Ajustar tamaño si es necesario
            actual_height = y_end - y_start
            actual_width = x_end - x_start
            
            if actual_height > 0 and actual_width > 0:
                # Asegurar que los datos tengan el tamaño correcto
                data_r_crop = data_r[:actual_height, :actual_width]
                data_g_crop = data_g[:actual_height, :actual_width]
                data_b_crop = data_b[:actual_height, :actual_width]
                
                # MEJORADO: Usar promedio en lugar de máximo para áreas solapadas
                # Esto crea transiciones más suaves entre campos
                mask_existing_r = mosaic_r[y_start:y_end, x_start:x_end] > 0
                mask_existing_g = mosaic_g[y_start:y_end, x_start:x_end] > 0
                mask_existing_b = mosaic_b[y_start:y_end, x_start:x_end] > 0
                
                # Donde ya hay datos, promediar. Donde no, colocar nuevos datos.
                mosaic_r[y_start:y_end, x_start:x_end] = np.where(
                    mask_existing_r,
                    (mosaic_r[y_start:y_end, x_start:x_end] + data_r_crop) / 2,
                    data_r_crop
                )
                mosaic_g[y_start:y_end, x_start:x_end] = np.where(
                    mask_existing_g,
                    (mosaic_g[y_start:y_end, x_start:x_end] + data_g_crop) / 2,
                    data_g_crop
                )
                mosaic_b[y_start:y_end, x_start:x_end] = np.where(
                    mask_existing_b,
                    (mosaic_b[y_start:y_end, x_start:x_end] + data_b_crop) / 2,
                    data_b_crop
                )
                
                print(f"   ✅ {campo} colocado en ({x_center}, {y_center}) - Tamaño: {actual_width}×{actual_height}")
            
        except Exception as e:
            print(f"   ❌ Error con {campo}: {e}")
            continue
    
    # Crear RGB
    from astropy.visualization import make_lupton_rgb
    
    print("🌈 Generando imagen RGB...")
    
    # Encontrar la región del mosaico que realmente tiene datos
    mask = (mosaic_r > 0) | (mosaic_g > 0) | (mosaic_b > 0)
    if np.any(mask):
        rows = np.any(mask, axis=1)
        cols = np.any(mask, axis=0)
        ymin, ymax = np.where(rows)[0][[0, -1]]
        xmin, xmax = np.where(cols)[0][[0, -1]]
        
        # Añadir margen
        margin = 100
        ymin = max(0, ymin - margin)
        ymax = min(mosaic_size, ymax + margin)
        xmin = max(0, xmin - margin)
        xmax = min(mosaic_size, xmax + margin)
        
        mosaic_r_crop = mosaic_r[ymin:ymax, xmin:xmax]
        mosaic_g_crop = mosaic_g[ymin:ymax, xmin:xmax]
        mosaic_b_crop = mosaic_b[ymin:ymax, xmin:xmax]
        
        print(f"📏 Región recortada: {mosaic_r_crop.shape}")
        
        rgb_image = make_lupton_rgb(mosaic_r_crop, mosaic_g_crop, mosaic_b_crop,
                                   Q=10, stretch=0.4)
        
        # Mostrar
        fig, ax = plt.subplots(figsize=(14, 12), facecolor='black')
        ax.imshow(rgb_image, origin='lower')
        ax.axis('off')
        
        # Añadir etiquetas en las posiciones recortadas
        for campo, (x, y) in posiciones.items():
            x_adj = x - xmin
            y_adj = y - ymin
            if 0 <= x_adj < rgb_image.shape[1] and 0 <= y_adj < rgb_image.shape[0]:
                ax.text(x_adj, y_adj, campo, color='yellow', fontsize=12, 
                       weight='bold', ha='center', va='center',
                       bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
        
        # Añadir barra de escala aproximada
        scale_deg = 1.0  # 1 grado
        # Estimación: 1 grado ≈ 200 píxeles en este mosaico
        scale_pixels = 200
        ax.plot([50, 50 + scale_pixels], [50, 50], 
                color='yellow', linewidth=4)
        ax.text(50 + scale_pixels/2, 30, f'{scale_deg}°', 
                color='yellow', ha='center', va='top', fontsize=12, weight='bold')
        
        plt.title(f'Centaurus A - Mosaico con Coordenadas Reales\n'
                 f'{len(coordenadas)} campos • Solapamiento continuo',
                 color='white', size=16, pad=20)
        
        output_path = '../anac_data/Figs-images/mosaico_coordenadas_reales.png'
        plt.savefig(output_path, dpi=250, bbox_inches='tight', facecolor='black')
        print(f"✅ Mosaico con coordenadas reales guardado: {output_path}")
        plt.show()
        
        return rgb_image
    else:
        print("❌ No hay datos en el mosaico")
        return None

In [ ]:
# Ejecutar el mosaico estilo Aladin
mosaico_continuo = crear_mosaico_aladin_style()